In [1]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [3]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 45.3 minutes


# **Part I. Describe: How bad is it, really?**
## **Part I. a. Percentage of late orders.**
### The function below calculates the percentage of orders delivered later than the promised delivery time. It also supports "all" for the zone, the time block, or both. The function is tested using the current 45 minutes promise for all four required cases.

In [4]:
def late_order_percentage(promise, zone, time_block):
    all_times = []

    selected_zones = ZONES if zone == "all" else [zone]
    selected_time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    for selected_zone in selected_zones:
      for selected_time_block in selected_time_blocks:
        times = delivery_times(selected_zone,selected_time_block,promise)
        all_times.extend(times)

    all_times = np.array(all_times)

    late_orders = np.sum(all_times > promise)
    total_orders = len(all_times)
    late_percentage = late_orders / total_orders * 100
    return late_percentage

print("- Specific zone and time block:", round(late_order_percentage(45, "Far West", "Fri/Sat eve"), 2),"%")
print("- All zones and one time block:", round(late_order_percentage(45, "all", "Lunch"), 2),"%")
print("- One zone and all time blocks:", round(late_order_percentage(45, "Far West", "all"), 2),"%")
print("- All zones and all time blocks:", round(late_order_percentage(45, "all", "all"), 2),"%")

- Specific zone and time block: 40.48 %
- All zones and one time block: 6.49 %
- One zone and all time blocks: 19.65 %
- All zones and all time blocks: 10.95 %


## **Part I. b. Ranking by late arrival rate**
### Using the current 45 minutes promise, the following analysis calculates the late arrival rate for every combination of zone and time block. The results are ranked from the highest late arrival rate to the lowest.

In [5]:
late_arrival_rate_ranking = []
for zone in ZONES:
    for time_block in TIME_BLOCKS:
        late_rate = late_order_percentage(45, zone, time_block)
        late_arrival_rate_ranking.append((zone, time_block, late_rate))
late_arrival_rate_ranking.sort(key=lambda result: result[2], reverse=True)
for rank, result in enumerate(late_arrival_rate_ranking, start=1):
  zone, time_block, late_rate = result
  print(rank,". ",zone,",",time_block, ":", round(late_rate,2),"%")

1 .  Far West , Fri/Sat eve : 40.95 %
2 .  North , Fri/Sat eve : 26.0 %
3 .  Far West , Weekday eve : 18.24 %
4 .  Central , Fri/Sat eve : 13.16 %
5 .  North , Weekday eve : 10.0 %
6 .  Far West , Other : 8.33 %
7 .  Far West , Lunch : 6.67 %
8 .  North , Lunch : 5.79 %
9 .  Central , Weekday eve : 5.45 %
10 .  North , Other : 5.0 %
11 .  Central , Other : 4.74 %
12 .  Central , Lunch : 2.31 %


## **Part I. c. Average delivery time**
### The function below calculates the average delivery time for a given delivery promise, zone, and time block. It also supports all for the zone, the time block, or both. The function is tested using the current 45 minutes promise for all four required cases.

In [6]:
def average_delivery_time(promise, zone, time_block):
    all_times = []

    selected_zones = ZONES if zone == "all" else [zone]
    selected_time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    for selected_zone in selected_zones:
      for selected_time_block in selected_time_blocks:
        times = delivery_times(selected_zone,selected_time_block,promise)
        all_times.extend(times)

    all_times = np.array(all_times)
    average_time = np.mean(all_times)
    return average_time

print("- Specific zone and time block:", round(average_delivery_time(45, "Far West", "Fri/Sat eve"), 2),"minutes")
print("- All zones and one time block:", round(average_delivery_time(45, "all", "Lunch"), 2),"minutes")
print("- One zone and all time blocks:", round(average_delivery_time(45, "Far West", "all"), 2),"minutes")
print("- All zones and all time blocks:", round(average_delivery_time(45, "all", "all"), 2),"minutes")


- Specific zone and time block: 42.54 minutes
- All zones and one time block: 29.02 minutes
- One zone and all time blocks: 36.26 minutes
- All zones and all time blocks: 32.13 minutes


## **Part I. d. Ranking by average delivery time**
### Using the current 45 minutes promise, the following analysis calculates the laverage delivery time for every combination of zone and time block. The results are ranked from the highest average delivery time to the lowest.

In [7]:
average_time_ranking = []
for zone in ZONES:
    for time_block in TIME_BLOCKS:
        average_time = average_delivery_time(45, zone, time_block)
        average_time_ranking.append((zone, time_block, average_time))

average_time_ranking.sort(key=lambda result: result[2], reverse=True)
for rank, result in enumerate(average_time_ranking, start=1):
  zone, time_block, average_time = result

  print(rank,". ",zone,",",time_block, ":", round(average_time,2),"minutes")

1 .  Far West , Fri/Sat eve : 43.88 minutes
2 .  North , Fri/Sat eve : 38.41 minutes
3 .  Far West , Weekday eve : 35.53 minutes
4 .  North , Weekday eve : 32.88 minutes
5 .  Central , Fri/Sat eve : 32.03 minutes
6 .  Far West , Other : 31.98 minutes
7 .  Far West , Lunch : 30.9 minutes
8 .  North , Lunch : 30.34 minutes
9 .  Central , Weekday eve : 30.2 minutes
10 .  North , Other : 28.77 minutes
11 .  Central , Lunch : 28.28 minutes
12 .  Central , Other : 26.83 minutes


## **Part I. e. The more relevant measure**
### The late arrival rate is more relevant to Rosa's delivery promise decision than the average delivery time.
### The results show that the "Far West, Fri/Sat eve" combination has the longest average delivery time at 43.88 minutes that is less than 45 minutes promise, but it has the highest late arrival rate at 40.95%. This indicates that an average can hide a large number of orders that exceed the promised time.
### Average delivery time is useful for understanding general operational efficiency. However, the late arrival rate directly measures how often Rosa’s fails to meet its promise. Because late orders can lead to refunds and customer churn.
### Therefore, the late arrival rate should be the primary measure for Rosa’s delivery promise decision. Average delivery time can be used as a supporting measure.

# **Part II. Exploring the best promise**
## **Part II. a. Total cost per late order**
### Each late order creates two types of costs: the direct refund and the profit lost because of customer churn. The function below calculates the total cost per late order using the figures provided in “COSTS”.

In [8]:
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [9]:
def cost_per_late_order(costs):
  refund_cost = costs["refund"]
  churn_cost = costs["churn_orders"]
  margin_cost = costs["margin"]

  total_cost = refund_cost + churn_cost * margin_cost
  return total_cost

print(f"Total cost per late order: ${cost_per_late_order(COSTS)}")

Total cost per late order: $26.2


## **Part II. b. Choosing the best promise**
### I tested promised delivery times from 5 to 95 minutes in 5-minute steps. For the Far West zone during the Fri/Sat evening time block, the recommended promise is 60 minutes, with the highest net profit of $1,010.40. Shorter promises attract more orders but create higher late order costs, while longer promises reduce late orders but also reduce demand. Therefore, 60 minutes provides the best balance between the number of orders and late order costs.

In [10]:
def choose_best_promise(zone, time_block, promises, costs):
  best_promise = None
  best_net_profit = - np.inf
  all_results = []

  late_order_costs = cost_per_late_order(costs)

  for promise in promises:
    times = np.array(delivery_times(zone, time_block, promise,seed=77))
    total_orders = len(times)
    late_orders = np.sum(times > promise)
    profit_from_orders = total_orders * costs["margin"]
    total_late_cost = late_orders * late_order_costs
    net_profit = profit_from_orders - total_late_cost
    all_results.append((promise, total_orders, late_orders, net_profit))
    if net_profit > best_net_profit:
      best_net_profit = net_profit
      best_promise = promise

  return best_promise, best_net_profit, all_results

promises_to_try = list(range(5, 100, 5))
best_promise, best_net_profit, results = choose_best_promise("Far West", "Fri/Sat eve", promises_to_try,COSTS)
for promise, total_orders, late_orders, net_profit in results:
  print(f"Promise: {promise} minutes")
  print(f"Total orders: {total_orders}")
  print(f"Late orders: {late_orders}")
  print(f"Net profit: ${net_profit:.2f}")

print(f"Best promise: {best_promise} minutes")
print(f"Best net profit: ${best_net_profit:.2f}")

Promise: 5 minutes
Total orders: 242
Late orders: 242
Net profit: $-4162.40
Promise: 10 minutes
Total orders: 242
Late orders: 242
Net profit: $-4162.40
Promise: 15 minutes
Total orders: 241
Late orders: 241
Net profit: $-4145.20
Promise: 20 minutes
Total orders: 241
Late orders: 240
Net profit: $-4119.00
Promise: 25 minutes
Total orders: 239
Late orders: 230
Net profit: $-3875.00
Promise: 30 minutes
Total orders: 237
Late orders: 208
Net profit: $-3316.60
Promise: 35 minutes
Total orders: 232
Late orders: 175
Net profit: $-2497.00
Promise: 40 minutes
Total orders: 224
Late orders: 130
Net profit: $-1390.00
Promise: 45 minutes
Total orders: 210
Late orders: 73
Net profit: $-22.60
Promise: 50 minutes
Total orders: 188
Late orders: 41
Net profit: $617.80
Promise: 55 minutes
Total orders: 158
Late orders: 16
Net profit: $1002.80
Promise: 60 minutes
Total orders: 121
Late orders: 3
Net profit: $1010.40
Promise: 65 minutes
Total orders: 84
Late orders: 2
Net profit: $703.60
Promise: 70 minu